In [4]:
import ndjson
import pandas as pd

In [5]:
with open("../../../../output_data/scraped_data/scraped_data_2023-03-10_0-1000000.ndjson") as f:
    d = ndjson.load(f)

In [6]:
len(d)

52555

In [35]:
x = [(_["id"], _["text"]) for _ in d if _["level"] == "0"]
len(x)

1666

In [52]:
a = pd.DataFrame(x)
a["len"] = a[1].str.len()
a = a.drop_duplicates(subset=[1])
a = a.reset_index(drop=True)
a

,0,1,len
0,01125149,Web Server's Default Page\nWeb Server's Defaul...,772
1,05008656,Lunchroom Blocks Zwolle - Lunchen in Hartje Ce...,1375
2,01071763,Koerier Friesland - Spoedkoerier Speedcargo Ko...,1800
3,04064715,M & D Holland te Emmen\nM&D Holland is een pro...,718
4,08034622,is functioning normally,23
...,...,...,...
1558,82833176,Webshop | Fit For Fertility\nAlle producten\nO...,6562
1559,82832811,sloten-vervangen.nl\nhome\nover ons\ncilinders...,3558
1560,82668043,de verbouwmeesters\nMeteen naar de inhoud\nHom...,1638
1561,82993998,Uncle Sam’s Burgers – Best Burgers in Town.\nS...,3794


In [84]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


vectorizer = TfidfVectorizer(max_df=0.05, min_df=0.001)
tfidf_matrix = vectorizer.fit_transform(a[1])
print(tfidf_matrix.shape)

# Calculate cosine similarity between all pairs of websites
similarity_matrix = cosine_similarity(tfidf_matrix)

(1563, 21917)


In [85]:
import numpy as np

labeled_dataset = []
for i in range(len(a)):
    # If the website has at least one other website with a similarity above the threshold,
    # label it as automatically generated content (1), otherwise label it as original content (0).
    similar_websites = np.where(similarity_matrix[i] >= 0.8)[0]
    label = 1 if len(similar_websites) > 1 else 0
    labeled_dataset.append((label, len(similar_websites)))


In [86]:
a["label"], a["sim_sites"] = zip(*labeled_dataset)
a

,0,1,len,label,sim_sites
0,01125149,Web Server's Default Page\nWeb Server's Defaul...,772,1,2
1,05008656,Lunchroom Blocks Zwolle - Lunchen in Hartje Ce...,1375,0,1
2,01071763,Koerier Friesland - Spoedkoerier Speedcargo Ko...,1800,0,1
3,04064715,M & D Holland te Emmen\nM&D Holland is een pro...,718,0,1
4,08034622,is functioning normally,23,1,2
...,...,...,...,...,...
1558,82833176,Webshop | Fit For Fertility\nAlle producten\nO...,6562,0,1
1559,82832811,sloten-vervangen.nl\nhome\nover ons\ncilinders...,3558,0,1
1560,82668043,de verbouwmeesters\nMeteen naar de inhoud\nHom...,1638,0,1
1561,82993998,Uncle Sam’s Burgers – Best Burgers in Town.\nS...,3794,0,1


In [70]:
np.where(similarity_matrix[125] >= 0.7)[0]

array([125, 393])

In [60]:
a.loc[393, 1]

"VDL Zekerheid\nMenu\nHome\nOver ons\nWie zijn wij\nWat doen wij\nOnze dienstverlening\nOnze kwaliteitswaarborgen\nLinks\nParticulier\nPensioen\nHypotheek\nSparen\nKrediet\nFinancial planning\nUitvaart\nZorg\nWonen\nRecht\nVerkeer\nRecreatie\nLinks\nZakelijk\nOndernemer\nWerknemerspensioen\nEmployee benefits\nRisico analyse\nVerzekeringen\nLinks\nNieuwe situatie\nUw gezin\nSamenwonen en trouwen\nKinderen\nScheiden\nOverlijden\nUw werk en inkomen\nNieuwe baan\nPromotie\nOntslag\nZiek zijn\nArbeidsongeschikt raken\nEerder stoppen met werken\nMet pensioen gaan\nEigen zaak starten\nBelasting betalen\nUw woning\nHuis kopen\nHuis huren\nVerbouwen\nVerhuizen\nJe studie\nStudiefinanciering\nStudieschuld\nOp kamers gaan\nEchtscheiding\nKlantenservice\nPremie berekenen\nOfferte aanvragen\nWijzigingen doorgeven\nHandige rekentools\nSchade melden\nAdreswijziging\nBel-mij-service\nTotaalrelatie-service\nKlanttevredenheid\nAanbeveling\nKlachtmelding\nDownloads\nNieuws\nParticulier\nZakelijk\nAanmeld

In [87]:
a.loc[a["label"]==1]

,0,1,len,label,sim_sites
0,01125149,Web Server's Default Page\nWeb Server's Defaul...,772,1,2
4,08034622,is functioning normally,23,1,2
7,01142382,Apache is functioning normally,30,1,2
8,01182889,Secured Home of steenzetterijligthart.nl\nWelc...,149,1,12
122,09089448,Web Server's Default Page\nWeb Server's Defaul...,771,1,2
231,17078601,Op deze domeinnaam is nog geen website geconfi...,108,1,2
240,17131051,jumbodelaak.nl\njumbodelaak.nl\nSomething amaz...,196,1,2
254,17165888,Secured Home of e-scoop.com\nWelcome to e-scoo...,123,1,12
320,20099204,Secured Home of primos.eu\nWelcome to primos.e...,119,1,11
325,20131739,Secured Home of loodet.nl\nWelcome to loodet.n...,119,1,12


In [31]:
np.where(similarity_matrix[1] >= 0.9)[0]

array([ 1, 81])